# Notebook 3 – Machine Learning Models

Three ML tasks on the toy store dataset:

| # | Task | Algorithm |
|---|------|-----------|
| 1 | Customer Segmentation | K-Means on RFM features |
| 2 | Monthly Revenue Forecasting | RandomForestRegressor |
| 3 | Order Cancellation Prediction | Logistic Regression |

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_or_generate, build_master
from src.ml_models import (
    CustomerSegmentation,
    RevenueForecaster,
    CancellationPredictor,
    build_rfm,
)

tables = load_or_generate('../data', seed=42)
master = build_master(tables)
print('Dataset ready –', len(master), 'rows')

---
## Part 1 – Customer Segmentation (K-Means / RFM)

We compute **Recency**, **Frequency**, and **Monetary** value per customer,
scale the features, and group customers into 4 segments.

In [ ]:
# --- RFM table preview ---
rfm = build_rfm(master)
print('RFM table shape:', rfm.shape)
display(rfm.describe().round(2))

In [ ]:
seg = CustomerSegmentation(n_clusters=4, random_state=42)
seg.fit(master)

print(f'Silhouette Score: {seg.silhouette_score_:.4f}')
print('\nSegment Summary (mean RFM):')
display(seg.segment_summary())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
seg.plot_segments(ax=ax)
plt.tight_layout()
plt.savefig('../data/fig_customer_segments.png', dpi=150)
plt.show()

In [ ]:
# Determine optimal k using the Elbow method
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

rfm_vals = rfm[['recency', 'frequency', 'monetary']]
X_scaled = StandardScaler().fit_transform(rfm_vals)

inertias = []
k_range = range(2, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(k_range), inertias, marker='o', linewidth=2)
ax.set_title('Elbow Method – Optimal Number of Clusters', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.savefig('../data/fig_elbow.png', dpi=150)
plt.show()

---
## Part 2 – Revenue Forecasting (RandomForestRegressor)

We aggregate revenue to monthly level, engineer lag and rolling features,
and train a RandomForestRegressor to predict future monthly revenue.

In [ ]:
forecaster = RevenueForecaster(n_estimators=200, random_state=42)
forecaster.fit(master, test_size=0.25)

metrics = forecaster.evaluate()
print(f"MAE : ${metrics['mae']:,.2f}")
print(f"R²  : {metrics['r2']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
forecaster.plot_forecast(ax=ax)
plt.tight_layout()
plt.savefig('../data/fig_revenue_forecast.png', dpi=150)
plt.show()

In [ ]:
import seaborn as sns

fi = forecaster.feature_importances().reset_index()
fi.columns = ['feature', 'importance']

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=fi, x='importance', y='feature', ax=ax, orient='h')
ax.set_title('Feature Importances – Revenue Forecaster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/fig_feature_importance.png', dpi=150)
plt.show()

---
## Part 3 – Cancellation Prediction (Logistic Regression)

We predict whether an order will be cancelled using product price, quantity,
customer age, order month, and product category.

In [ ]:
predictor = CancellationPredictor(C=1.0, random_state=42)
predictor.fit(master, test_size=0.2)

report = predictor.evaluate()
print('Classification Report:')
display(pd.DataFrame(report).transpose().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
predictor.plot_coefficients(ax=ax)
plt.tight_layout()
plt.savefig('../data/fig_cancel_coefficients.png', dpi=150)
plt.show()

---
## Summary

| Model | Key Metric |
|-------|------------|
| K-Means Customer Segmentation | Silhouette Score |
| RandomForest Revenue Forecaster | MAE & R² |
| Logistic Regression Cancellation | Precision / Recall / F1 |